# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Dataset identifier: {metadata.identifier}")
print(f"Dataset published: {metadata.datePublished}\n")
# Display keywords and use cases
print("Keywords:", getattr(metadata, 'keywords', []))
print("Use cases:", getattr(metadata, 'dataUseCases', []))

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema includes potentially multiple record sets. We'll list all available `@id`s for record sets, and for each record set, the fields and their `@id`s.

In [ ]:
# List all available record sets
record_sets = list(dataset.record_sets)
print("Available Record Sets (@id):")
for record_set in record_sets:
    print(f"- {record_set['@id']}: {record_set.get('name','')}")

# For each record set, list field @ids
for record_set in record_sets:
    print(f"\nFields for Record Set {record_set['@id']}:")
    fields = record_set.get('fields', [])
    for field in fields:
        print(f"    - Field @id: {field['@id']}, name: {field.get('name','')}, dataType: {field.get('dataType','')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.
Below, all record sets are loaded into DataFrames for convenience.

In [ ]:
# Extract data from each record set
# List all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\n{record_set_id}: DataFrame columns:")
    print(df.columns.tolist())
    print(df.head(3))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For demonstration, we select the main survey record set (usually the largest one).

In [ ]:
# Pick the primary record set for further analysis
# Usually the main record set contains fields like 'GAD-7_score', 'PHQ-9_score', etc.

# Heuristic: choose the record set with the most rows
main_rs_id = max(dataframes.keys(), key=lambda k: len(dataframes[k]))

df = dataframes[main_rs_id]
print(f"Using record set @id: {main_rs_id}\nColumns: {df.columns.tolist()}")

# Identify numeric fields by dataType
fields_info = [f for rs in record_sets if rs['@id']==main_rs_id for f in rs.get('fields',[])]
numeric_fields = [f['@id'] for f in fields_info if f.get('dataType','').lower() in ['integer','float','number']]
print(f"Numeric fields (@id): {numeric_fields}")

# For demonstration, use first available numeric field (e.g., GAD-7_score @id)
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    numeric_field_id = df.select_dtypes(include='number').columns[0]
print(f"Using numeric_field_id: {numeric_field_id}")

# Show summary statistics
print("Summary statistics:")
print(df[numeric_field_id].describe())

# Filter records with numeric_field > threshold
threshold = df[numeric_field_id].mean() + df[numeric_field_id].std()
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field: pick first with dataType Text or String
cat_fields = [f['@id'] for f in fields_info if f.get('dataType','').lower() in ['text','string']]
if cat_fields:
    group_field_id = cat_fields[0]
    print(f"Grouping by: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print("\nGrouped data:")
    print(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of the selected numeric field, and show the average per group (if grouping was possible).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id], kde=True, bins=15)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If group_field_id and grouped_df exist, plot group averages
if 'group_field_id' in locals() and 'grouped_df' in locals():
    plt.figure(figsize=(8,5))
    sns.barplot(
        x=grouped_df[group_field_id],
        y=grouped_df[numeric_field_id])
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded metadata and survey data from Kilifi County FAIR² dataset via Croissant schema.
- Identified record sets and fields by `@id` for reproducible data processing.
- Performed exploratory analysis on a key numeric indicator, filtered and normalized values, and grouped by a demographic field.
- Visualizations show distribution and group-level variation, facilitating rapid insight into mental health patterns.
- This notebook framework can be adapted to other Croissant-based datasets for FAIR and AI-ready workflows.